# Eclipse — HalfKAv2 WDL Training

Trains the Eclipse chess engine's NNUE on game-outcome (W/D/L) labels.

**Inputs**
* Kaggle Dataset: `eclipse-wdl-training` containing `wdl_training.txt` or `wdl_training.txt.gz`.
  Lines are `<fen>;<W>;<D>;<L>` (preferred) or legacy `<fen>;<score_cp>`.
* Hardware: GPU T4 ×1 (Settings → Accelerator). P100 works the same; T4 is usually queued faster.

**Output**
* `/kaggle/working/halfkav2.pt` — PyTorch state_dict in the layout `scripts/convert_halfkav2_nnue.py from-torch` expects. Download locally and pack into `data/eclipse.nnue`.

**Source of truth**
This notebook embeds a copy of `scripts/train_halfkav2.py`. When the trainer changes in a way that affects the trained net, update both. See `KAGGLE.md` in the repo for the full pipeline.

In [ ]:
# Environment check
import sys, platform
import torch
print(f'python   {platform.python_version()}')
print(f'torch    {torch.__version__}')
print(f'cuda     {torch.cuda.is_available()} ({torch.version.cuda})')
if torch.cuda.is_available():
    print(f'gpu      {torch.cuda.get_device_name(0)}')
    print(f'vram     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Locate dataset on /kaggle/input/. Decompress wdl_training.txt.gz into
# /kaggle/working/ if only the gzip is present (faster to read uncompressed).
import gzip, shutil, os
from pathlib import Path

DATASET_SLUG = 'eclipse-partial'   # Kaggle dataset: simbae11/eclipse-partial
INPUT_DIR    = Path(f'/kaggle/input/{DATASET_SLUG}')
WORK_DIR     = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)

txt_in_input = INPUT_DIR / 'wdl_training.txt'
gz_in_input  = INPUT_DIR / 'wdl_training.txt.gz'
txt_in_work  = WORK_DIR / 'wdl_training.txt'

if txt_in_input.exists():
    DATA_PATH = txt_in_input
elif gz_in_input.exists():
    if not txt_in_work.exists():
        print(f'decompressing {gz_in_input} -> {txt_in_work}')
        with gzip.open(gz_in_input, 'rb') as src, open(txt_in_work, 'wb') as dst:
            shutil.copyfileobj(src, dst, length=1<<24)
    DATA_PATH = txt_in_work
else:
    raise FileNotFoundError(
        f'No wdl_training.txt(.gz) under {INPUT_DIR}. Attach the dataset via '
        f'Add data, or correct DATASET_SLUG.')

size_mb = DATA_PATH.stat().st_size / 1e6
print(f'training data: {DATA_PATH} ({size_mb:.1f} MB)')

In [ ]:
# HalfKAv2 feature indexing -- exact mirror of src/nnue.cpp::feature_index
# and scripts/train_halfkav2.py. Drift here = silently wrong inference.
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

FT_IN_FEATURES = 45056          # HalfKAv2: 64 (king sq) * 64 (piece sq) * 11 (piece slots)
FT_OUT         = 1024           # keep in sync with kFtOutSize in src/nnue.hpp
L1_IN          = 2 * FT_OUT     # 2048 (concat of both perspectives)
L1_OUT         = 256
L2_OUT         = 64
L3_OUT         = 3              # WDL head: [Win, Draw, Loss]
N_PIECE_SLOTS  = 11

_PIECE_LETTER_TO_TYPE = {'p':1, 'n':2, 'b':3, 'r':4, 'q':5, 'k':6}
_FLIP_RANK = lambda s: s ^ 56

def parse_fen_board(fen_board):
    pieces = []
    sq = 56
    for ch in fen_board:
        if ch == '/':
            sq -= 16
            continue
        if ch.isdigit():
            sq += int(ch)
            continue
        is_white = ch.isupper()
        pt = _PIECE_LETTER_TO_TYPE[ch.lower()]
        pieces.append((sq, 0 if is_white else 1, pt))
        sq += 1
    return pieces

def feature_index(king_sq, piece_sq, pt, piece_is_ours):
    # ours: P=0,N=1,B=2,R=3,Q=4 (own king skipped). theirs: P=5..K=10.
    if piece_is_ours and pt == 6:
        raise ValueError('own king is not a HalfKAv2 feature')
    slot = (pt - 1) if piece_is_ours else (5 + (pt - 1))
    return king_sq * (64 * N_PIECE_SLOTS) + piece_sq * N_PIECE_SLOTS + slot

def fen_to_features(fen):
    parts = fen.split()
    board, stm = parts[0], parts[1]
    pieces = parse_fen_board(board)
    stm_color = 0 if stm == 'w' else 1
    king_sq = [None, None]
    for sq, color, pt in pieces:
        if pt == 6:
            king_sq[color] = sq
    if king_sq[0] is None or king_sq[1] is None:
        raise ValueError(f'FEN missing a king: {fen}')
    indices = [[], []]
    for persp in (0, 1):
        ksq = king_sq[persp] if persp == 0 else _FLIP_RANK(king_sq[persp])
        for sq, color, pt in pieces:
            is_ours = (color == persp)
            if is_ours and pt == 6:
                continue
            psq = sq if persp == 0 else _FLIP_RANK(sq)
            indices[persp].append(feature_index(ksq, psq, pt, is_ours))
    return (indices[0], indices[1]) if stm_color == 0 else (indices[1], indices[0])

In [ ]:
# Dataset: flat numpy arrays, padded with FT_IN_FEATURES sentinel.
# 10M positions * (32+32) int32 + 10M*3 float32 = ~2.7 GB RAM.
from torch.utils.data import DataLoader, Dataset
MAX_ACTIVE = 32                 # HalfKAv2 has at most 31 active per perspective

class HalfKAv2Dataset(Dataset):
    def __init__(self, path, max_lines=None, cp_scale=410.0):
        # Two-pass: count rows, allocate, fill. Avoids Python list growth.
        n = 0
        with open(path) as f:
            for line in f:
                s = line.strip()
                if not s or s.startswith('#'):
                    continue
                n += 1
                if max_lines is not None and n >= max_lines:
                    break
        if n == 0:
            raise RuntimeError(f'no training rows in {path}')

        self.us_idx   = np.full((n, MAX_ACTIVE), FT_IN_FEATURES, dtype=np.int32)
        self.them_idx = np.full((n, MAX_ACTIVE), FT_IN_FEATURES, dtype=np.int32)
        self.targets  = np.zeros((n, 3), dtype=np.float32)

        i = 0; skipped = 0; n_wdl = 0; n_cp = 0
        import math
        with open(path) as f:
            for ln, line in enumerate(f, start=1):
                s = line.strip()
                if not s or s.startswith('#'):
                    continue
                if i >= n:
                    break
                parts = s.split(';')
                if len(parts) == 4:
                    try:
                        fen = parts[0].strip()
                        w, d, l = (float(parts[1]), float(parts[2]), float(parts[3]))
                    except ValueError:
                        skipped += 1; continue
                    target = (w, d, l); n_wdl += 1
                elif len(parts) == 2:
                    try:
                        fen = parts[0].strip()
                        cp  = float(parts[1])
                    except ValueError:
                        skipped += 1; continue
                    pw = 1.0 / (1.0 + math.exp(-cp / cp_scale))
                    target = (pw, 0.0, 1.0 - pw); n_cp += 1
                else:
                    skipped += 1; continue
                try:
                    us, them = fen_to_features(fen)
                except (KeyError, IndexError, ValueError):
                    skipped += 1; continue
                k_us   = min(len(us),   MAX_ACTIVE)
                k_them = min(len(them), MAX_ACTIVE)
                self.us_idx[i,   :k_us]   = us[:k_us]
                self.them_idx[i, :k_them] = them[:k_them]
                self.targets[i] = target
                i += 1
                if i % 500_000 == 0:
                    print(f'  loaded {i}/{n}')

        if i < n:
            self.us_idx   = self.us_idx[:i]
            self.them_idx = self.them_idx[:i]
            self.targets  = self.targets[:i]
        print(f'loaded {i} positions ({skipped} skipped, wdl={n_wdl}, cp={n_cp})')
        if n_cp > 0 and n_wdl == 0:
            print('WARNING: all rows are legacy cp format. Result will be a '
                  'W/L binary classifier (draw channel is zero).')

    def __len__(self):  return len(self.targets)
    def __getitem__(self, idx):
        return self.us_idx[idx], self.them_idx[idx], self.targets[idx]

def collate(batch):
    us   = np.stack([b[0] for b in batch])
    them = np.stack([b[1] for b in batch])
    tgt  = np.stack([b[2] for b in batch])
    return (torch.from_numpy(us).long(),
            torch.from_numpy(them).long(),
            torch.from_numpy(tgt))

In [ ]:
# Model -- float mirror of the C++ inference path in src/nnue.cpp.
class ClippedReLU(nn.Module):
    def forward(self, x):
        return torch.clamp(x, 0.0, 1.0)

class HalfKAv2Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Last embedding row is the padding sentinel: zeroed at init,
        # pinned to zero after every step so padded slots contribute 0.
        self.ft = nn.Embedding(FT_IN_FEATURES + 1, FT_OUT,
                               padding_idx=FT_IN_FEATURES)
        nn.init.uniform_(self.ft.weight, -0.01, 0.01)
        with torch.no_grad():
            self.ft.weight[FT_IN_FEATURES].zero_()
        self.ft_bias = nn.Parameter(torch.zeros(FT_OUT))
        self.l1  = nn.Linear(L1_IN, L1_OUT)
        self.l2  = nn.Linear(L1_OUT, L2_OUT)
        self.l3  = nn.Linear(L2_OUT, L3_OUT)
        self.act = ClippedReLU()

    def forward(self, us_idx, them_idx):
        us   = self.ft(us_idx).sum(dim=1)   + self.ft_bias
        them = self.ft(them_idx).sum(dim=1) + self.ft_bias
        x = torch.cat([self.act(us), self.act(them)], dim=1)
        x = self.act(self.l1(x))
        x = self.act(self.l2(x))
        return self.l3(x)

    def export_state_dict_for_quantization(self):
        # Repack the Embedding into a Linear-style weight matrix
        # [FT_OUT, FT_IN_FEATURES] so convert_halfkav2_nnue.py reads it cleanly.
        sd = {}
        ft_w = self.ft.weight.detach()
        sd['ft.weight'] = ft_w[:FT_IN_FEATURES].T.contiguous()
        sd['ft.bias']   = self.ft_bias.detach()
        sd['l1.weight'] = self.l1.weight.detach()
        sd['l1.bias']   = self.l1.bias.detach()
        sd['l2.weight'] = self.l2.weight.detach()
        sd['l2.bias']   = self.l2.bias.detach()
        sd['l3.weight'] = self.l3.weight.detach()
        sd['l3.bias']   = self.l3.bias.detach()
        return sd

In [ ]:
# Training config. Defaults match scripts/train_halfkav2.py.
CONFIG = dict(
    epochs       = 5,
    batch_size   = 4096,         # raise to 8192/16384 on P100 if VRAM allows
    lr           = 1e-3,
    weight_decay = 1e-4,
    warmup_steps = 500,
    val_frac     = 0.05,
    num_workers  = 2,            # Kaggle has 2 CPU cores in GPU sessions
    max_lines    = None,         # set to a small int for a quick smoke run
    cp_scale     = 410.0,        # also used as output_cp_per_unit in the .nnue
    out_path     = '/kaggle/working/halfkav2.pt',
)
for k, v in CONFIG.items():
    print(f'  {k:14s} = {v}')

In [ ]:
# Training loop. Soft-label cross-entropy on log-softmax. Adam + linear warmup
# + cosine schedule. 5% deterministic val hold-out (last slice), so val rows
# never leak into train across reruns.
import time
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'training on {device}')

t0 = time.time()
full_ds = HalfKAv2Dataset(DATA_PATH, max_lines=CONFIG['max_lines'],
                           cp_scale=CONFIG['cp_scale'])
print(f'dataset loaded in {time.time()-t0:.1f}s')

n_total = len(full_ds)
n_val   = max(1024, int(n_total * CONFIG['val_frac']))
n_val   = min(n_val, n_total // 10)
n_train = n_total - n_val
print(f'split: {n_train} train / {n_val} val')
train_ds = torch.utils.data.Subset(full_ds, range(n_train))
val_ds   = torch.utils.data.Subset(full_ds, range(n_train, n_total))

loader     = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True,
                        num_workers=CONFIG['num_workers'], collate_fn=collate,
                        drop_last=True, pin_memory=(device.type == 'cuda'))
val_loader = DataLoader(val_ds,   batch_size=CONFIG['batch_size'], shuffle=False,
                        num_workers=max(1, CONFIG['num_workers']//2),
                        collate_fn=collate, drop_last=False,
                        pin_memory=(device.type == 'cuda'))

net    = HalfKAv2Net().to(device)
opt    = torch.optim.Adam(net.parameters(), lr=CONFIG['lr'],
                          weight_decay=CONFIG['weight_decay'])
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG['epochs'])

def soft_label_ce(logits, target):
    return -(target * F.log_softmax(logits, dim=1)).sum(dim=1).mean()

step = 0
for epoch in range(CONFIG['epochs']):
    net.train()
    ep_loss = 0.0; nbatch = 0
    for us, them, target in loader:
        us, them, target = us.to(device), them.to(device), target.to(device)
        if step < CONFIG['warmup_steps']:
            scale = (step + 1) / max(1, CONFIG['warmup_steps'])
            for g in opt.param_groups:
                g['lr'] = CONFIG['lr'] * scale
        logits = net(us, them)
        loss   = soft_label_ce(logits, target)
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            net.ft.weight[FT_IN_FEATURES].zero_()
        ep_loss += loss.item(); nbatch += 1; step += 1
        if step % 200 == 0:
            print(f'  epoch {epoch} step {step} loss {loss.item():.5f}')

    net.eval()
    vl_sum = 0.0; vacc = 0; vcount = 0
    with torch.no_grad():
        for us, them, target in val_loader:
            us, them, target = us.to(device), them.to(device), target.to(device)
            logits  = net(us, them)
            vl_sum += soft_label_ce(logits, target).item() * target.shape[0]
            vacc   += (logits.argmax(dim=1) == target.argmax(dim=1)).sum().item()
            vcount += target.shape[0]
    cosine.step()
    print(f'epoch {epoch} train_loss {ep_loss/max(nbatch,1):.5f} '
          f'val_loss {vl_sum/max(1,vcount):.5f} '
          f'val_argmax_acc {vacc/max(1,vcount):.3f} '
          f'(lr next: {cosine.get_last_lr()[0]:.6f})')

In [ ]:
# Save state_dict in the layout convert_halfkav2_nnue.py expects.
out_path = Path(CONFIG['out_path'])
out_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(net.export_state_dict_for_quantization(), out_path)
size_mb = out_path.stat().st_size / 1e6
print(f'saved {out_path} ({size_mb:.1f} MB)')
print()
print('Next: download halfkav2.pt and locally run')
print('  python scripts/convert_halfkav2_nnue.py from-torch \\')
print(f'      --state-dict data/halfkav2.pt \\')
print(f'      --out data/eclipse.nnue \\')
print(f'      --output-cp-per-unit {CONFIG["cp_scale"]}')

## Saving the artifact off Kaggle

1. Click **Save Version → Save & Run All**. The notebook output (including `/kaggle/working/halfkav2.pt`) is captured on the version that finishes successfully.
2. Open the saved version → **Output** tab → click `halfkav2.pt` → Download.
3. Locally, drop it into `data/` and run the convert command above. See `KAGGLE.md` for verification steps and saturation-warning interpretation.